In [2]:
import pandas as pd
import requests
import io

sheet_id = "1UXuXCuYDSM4LR6ofEf5MEbT2zDklDyItukczhkGDTGA"
gid = "349661045"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=tsv&gid={gid}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

try:
    print("Загрузка данных...")
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    
    df = pd.read_csv(io.StringIO(response.content.decode('utf-8')), sep='\t')
    df = df.iloc[:, :4]
    df.columns = ['ID Клиента', 'наименование', 'выручка', 'группа']
    print(f"Успешно загружено! Строк: {len(df)}")
except Exception as e:
    print(f"Ошибка загрузки: {e}")

Загрузка данных...
Успешно загружено! Строк: 1574


In [3]:
# 1. Тщательно очищаем столбец "выручка" от пробелов и запятых
df['выручка'] = df['выручка'].astype(str)
# Удаляем обычные пробелы, неразрывные веб-пробелы (\xa0) и меняем запятую на точку
df['выручка'] = df['выручка'].str.replace(' ', '').str.replace('\xa0', '').str.replace(',', '.')

# Превращаем в честные числа. Теперь ни одно число не потеряется!
df['выручка'] = pd.to_numeric(df['выручка'], errors='coerce')

# 2. Функция маппинга групп
def classify_product(name):
    name_lower = str(name).lower().strip()
    if any(word in name_lower for word in ['коврик', 'ковр', 'ковёр']):
        return 'коврики'
    elif any(word in name_lower for word in ['сигнал', 'охран', 'starline', 'сирена', 'обход', 'обх', 
                                            'диагностическ', 'прибора', 'цепи питания', 'управление', 'alfa comfort', 'rsa']):
        return 'сигнализация'
    elif any(word in name_lower for word in ['парков', 'парктр', 'датчик', 'park master', 'parkmaster']):
        return 'парктроник'
    elif any(word in name_lower for word in ['защит', 'картер', 'редуктор', ' защ']):
        return 'защита'
    elif any(word in name_lower for word in ['камер', 'камеы', 'з/в', 'видеокамер']):
        return 'камера заднего вида'
    return 'Не определено'

df['группа'] = df['наименование'].apply(classify_product)

# Проверяем, что в выручке больше нет пропусков (должно быть 0)
print(f"Пропущенных ячеек в выручке после очистки: {df['выручка'].isna().sum()}")

Пропущенных ячеек в выручке после очистки: 0


In [4]:
# Применяем функцию к каждой строке
df['группа'] = df['наименование'].apply(classify_product)

# Проверяем, остались ли неклассифицированные строки
unclassified = df[df['группа'].isna()]

print(f"Количество строк без группы: {len(unclassified)}")
if len(unclassified) > 0:
    print("Нераспределенные позиции:")
    print(unclassified['наименование'].unique())
else:
    print("Отлично! Все 100% данных успешно распределены по группам.")

Количество строк без группы: 0
Отлично! Все 100% данных успешно распределены по группам.


In [5]:
# Преобразуем выручку в числа, если она распозналась как текст
if df['выручка'].dtype == 'object':
    df['выручка'] = df['выручка'].astype(str).str.replace(',', '.')
df['выручка'] = pd.to_numeric(df['выручка'], errors='coerce')

# Считаем сумму математически
summary = df.groupby('группа').agg(
    Количество_сделок=('выручка', 'count'),
    Общая_выручка=('выручка', 'sum')
).sort_values(by='Общая_выручка', ascending=False).round(2)

summary

,Количество_сделок,Общая_выручка
группа,,
защита,756,233595.64
коврики,440,54659.05
сигнализация,114,16523.29
камера заднего вида,32,11303.40
парктроник,228,5792.72
Не определено,4,2716.30


In [5]:
output_filename = "ТЗ_аналитик_Выполнено.csv"

# Сохраняем результат
df.to_csv(output_filename, index=False, encoding='utf-8-sig', sep=',')
print(f"Файл сохранен как: {output_filename}")

Файл сохранен как: ТЗ_аналитик_Выполнено.csv
